# The latent space of a trained classifier

Take the penultimate layer's output as an embedding, project it to two dimensions, and see the structure the network built without ever being asked to.

**Runs on:** CPU — about 3 minutes &nbsp;·&nbsp; **Slides:** [Chapter 10 — Interpreting What ConvNets Learn](../../../course-web-slides/ch10/index.html) &nbsp;·&nbsp; **Section:** 04 — What the network has organised

---

## A classifier, and its penultimate layer

In [ ]:
import keras
from keras import layers
from keras.datasets import fashion_mnist
import numpy as np

(x, y), (xt, yt) = fashion_mnist.load_data()
x = x.reshape(-1, 28, 28, 1).astype("float32") / 255
xt = xt.reshape(-1, 28, 28, 1).astype("float32") / 255
classes = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]

inputs = keras.Input(shape=(28, 28, 1))
z = layers.Conv2D(32, 3, activation="relu")(inputs)
z = layers.MaxPooling2D(2)(z)
z = layers.Conv2D(64, 3, activation="relu")(z)
z = layers.MaxPooling2D(2)(z)
z = layers.Flatten()(z)
embedding = layers.Dense(32, activation="relu", name="embedding")(z)
outputs = layers.Dense(10, activation="softmax")(embedding)
model = keras.Model(inputs, outputs)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(x, y, epochs=6, batch_size=128, validation_split=.1, verbose=2)
print("test:", model.evaluate(xt, yt, verbose=0)[1])

## Extracting the embedding

In [ ]:
encoder = keras.Model(model.input, model.get_layer("embedding").output)
emb = encoder.predict(xt[:4000], verbose=0)
labels = yt[:4000]
print("embedding:", emb.shape)

Thirty-two numbers per garment. **The classifier was never asked to organise this space** — it was asked to get the label right, and the organisation is a by-product.

## Projecting it

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

pca = PCA(n_components=2).fit_transform(emb)
tsne = TSNE(n_components=2, init="pca", perplexity=30,
            random_state=0).fit_transform(emb)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 6.4))
for ax, proj, name in [(a1, pca, "PCA"), (a2, tsne, "t-SNE")]:
    sc = ax.scatter(proj[:, 0], proj[:, 1], c=labels, cmap="tab10", s=6, alpha=.75)
    ax.set_title(f"{name} of the 32-d embedding"); ax.set_xticks([]); ax.set_yticks([])
handles = [plt.Line2D([], [], marker="o", ls="", color=plt.cm.tab10(i / 9),
                      label=classes[i]) for i in range(10)]
fig.legend(handles=handles, loc="lower center", ncol=10, fontsize=8.5,
           frameon=False, bbox_to_anchor=(0.5, -0.03))
plt.tight_layout(); plt.show()

## Reading the structure

Three things are worth pointing out on the t-SNE panel.

**Footwear clusters together** — sandal, sneaker, and boot sit adjacent, because they *are* adjacent. Nobody encoded that relation.

**Shirt overlaps with T-shirt, pullover, and coat.** Those are genuinely confusable and the confusion matrix will show it. The geometry predicts the errors.

**Bag and trouser sit apart from everything.** Distinctive shapes, distinctive region.

## The geometry predicts the confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix

pred = model.predict(xt[:4000], verbose=0).argmax(axis=1)
cm = confusion_matrix(labels, pred, normalize="true")

plt.figure(figsize=(7.5, 6.4))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(10), classes, rotation=45, ha="right")
plt.yticks(range(10), classes)
plt.colorbar(label="fraction"); plt.title("Confusion matrix")
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0.03 and i != j:
            plt.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center", fontsize=7)
plt.tight_layout(); plt.show()

The off-diagonal mass sits exactly where the embedding clusters overlap. **The two pictures are the same fact seen twice**, and the embedding version tells you *why* — the model's representation does not separate those classes, so no decision boundary drawn on it can.

## Nearest neighbours in the learned space

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=6).fit(emb)
query_indices = [3, 17, 42, 77]

fig, axes = plt.subplots(len(query_indices), 6, figsize=(10, 1.8 * len(query_indices)))
for row, qi in enumerate(query_indices):
    _, idx = nn.kneighbors(emb[qi:qi+1])
    for col, j in enumerate(idx[0]):
        axes[row, col].imshow(xt[j, :, :, 0], cmap="gray_r")
        axes[row, col].axis("off")
        axes[row, col].set_title("query" if col == 0 else classes[labels[j]],
                                 fontsize=8)
plt.suptitle("Nearest neighbours in the 32-d embedding", y=1.0)
plt.tight_layout(); plt.show()

Similar garments, retrieved by distance in a space nobody designed for retrieval. **This is the same mechanism as chapter 15's embeddings and chapter 16's vector database** — a classifier trained on labels produces a usable similarity metric as a by-product, and that by-product is often more valuable than the labels.

---

## What to take away

- The penultimate layer is an embedding you can use for retrieval and clustering.
- Its structure is emergent — the model was optimised for labels, not geometry.
- Overlapping clusters predict the confusion matrix, and explain it.
- The same idea scales up to chapter 15's embeddings and chapter 16's vector databases.